In [15]:
import pandas as pd
import re
import glob
import os
from bs4 import BeautifulSoup
import os
import json
from typing import List, Dict, Tuple
import math
import re
from rank_bm25 import BM25Okapi
from collections import defaultdict

In [ ]:
## Finder Sections 
# Question Category	Count	Percentage
# Accounting	491	8.61% --> 
# Company Overview	1081	18.95%. --> Business
# Financials	990	17.36%
# Footnotes	953	16.71%
# Governance	718	12.59%
# Legal	490	8.59%
# Risk	490	8.59%
# Shareholder Return	490	8.59%
# Total	5703	100.00%

## SEC 10k Filings
#         "Business": "1",
    # "Risk Factors": "1A",
    # "Unresolved Staff Comments": "1B",
    # "Cybersecurity": "1C",
    # "Properties": "2",
    # "Legal Proceedings": "3",
    # "Mine Safety Disclosures": "4",
    # "Market for Registrants Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities": "5",
    # "Selected Financial Data (prior to February 2021)": "6",
    # "Managements Discussion and Analysis": "7",
    # "Quantitative and Qualitative Disclosures about Market Risk": "7A",
    # "Financial Statements and Supplementary Data": "8",
    # "Changes in and Disagreements with Accountants on Accounting and Financial Disclosure": "9",
    # "Controls and Procedures": "9A",
    # "Other Information": "9B",
    # "Directors, Executive Officers and Corporate Governance": "10",
    # "Executive Compensation": "11",
    # "Security Ownership of Certain Beneficial Owners and Management and Related Stockholder Matters": "12",
    # "Certain Relationships and Related Transactions, and Director Independence": "13",
    # "Principal Accountant Fees and Services": "14",
    # "Exhibits and Financial Statement Schedules": "15"

In [271]:
def extract_text(text, item_start, item_end):
    item_start = item_start
    item_end = item_end
    starts = [i.start() for i in item_start.finditer(text)]
    ends = [i.start() for i in item_end.finditer(text)]
    positions = list()
    for s in starts:
        control = 0
        for e in ends:
            if control == 0:
                if s < e:
                    control = 1
                    positions.append([s,e])
    item_length = 0
    item_position = list()
    for p in positions:
        if (p[1]-p[0]) > item_length:
            item_length = p[1]-p[0]
            item_position = p

    item_text = text[item_position[0]:item_position[1]]

    return(item_text)

In [273]:
def read_html_file(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    return text


def clean_text_basic(text: str) -> str:
    text = re.sub(r'page\s*\d+(\s*\|\s*sec.*)?', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

def lemmatize_text(text: str) -> str:
    if nlp is None:
        return text
    doc = nlp(text)
    lemmas = [token.lemma_ for token in doc if not token.is_space]
    return " ".join(lemmas)

def preprocess(text: str, lemmatize: bool = False) -> str:
    t = clean_text_basic(text)
    if lemmatize:
        t = lemmatize_text(t)
    return t

In [307]:
html_folder = "10k"
file_paths = glob.glob(os.path.join(html_folder, "*.html")) + glob.glob(os.path.join(html_folder, "*.htm"))

In [309]:
file_paths

['10k/META.html',
 '10k/CAH.html',
 '10k/PGR.html',
 '10k/MDLZ.html',
 '10k/DXCM.html',
 '10k/HPQ.html',
 '10k/TEL.html',
 '10k/BA.html',
 '10k/PNC.html',
 '10k/NOC.html',
 '10k/MOS.html',
 '10k/ALL.html',
 '10k/XEL.html',
 '10k/MRK.html',
 '10k/SO.html',
 '10k/WFC.html',
 '10k/AOS.html',
 '10k/TXT.html',
 '10k/GEN.html',
 '10k/NEM.html',
 '10k/CLX.html',
 '10k/NUE.html',
 '10k/LOW.html',
 '10k/AXP.html',
 '10k/COR.html',
 '10k/DTE.html',
 '10k/GS.html',
 '10k/KIM.html',
 '10k/TRV.html',
 '10k/AIG.html',
 '10k/RMD.html',
 '10k/ES.html',
 '10k/CMCSA.html',
 '10k/PEG.html',
 '10k/ORCL.html',
 '10k/VMC.html',
 '10k/NVDA.html',
 '10k/HAS.html',
 '10k/EQIX.html',
 '10k/WDC.html',
 '10k/NXPI.html',
 '10k/FANG.html',
 '10k/TRGP.html',
 '10k/VRSK.html',
 '10k/GRMN.html',
 '10k/ESS.html',
 '10k/IP.html',
 '10k/CMG.html',
 '10k/LW.html',
 '10k/BX.html',
 '10k/CDW.html',
 '10k/PH.html',
 '10k/EG.html',
 '10k/CCI.html',
 '10k/JBHT.html',
 '10k/AEE.html',
 '10k/WST.html',
 '10k/KDP.html',
 '10k/JBL

In [340]:
def section_text(text, section):
    ## Secion 1 Buisness Overview 
    if section == 1 or section == 0:
        try:
            item1_start = re.compile("item\s*[1][\.\;\:\-\_]*\s*\\b", re.IGNORECASE)
            item1_end = re.compile("item\s*1a[\.\;\:\-\_]\s*Risk|item\s*2[\.\,\;\:\-\_]\s*Prop", re.IGNORECASE)
            businessText = extract_text(text, item1_start, item1_end)
            return businessText
        except:
            businessText = "Something went wrong!"

    ## Secion 2 Risk
    if section == 2 or section == 0:
        try:
            item1a_start = re.compile("(?<!,\s)item\s*1a[\.\;\:\-\_]\s*Risk", re.IGNORECASE)
            item1a_end = re.compile("item\s*2[\.\;\:\-\_]\s*Prop|item\s*[1][\.\;\:\-\_]*\s*\\b", re.IGNORECASE)
            riskText = extract_text(text, item1a_start, item1a_end)
            return riskText
        except:
            riskText = "Something went wrong!"

    ## Secion 3 Management's Discussion and Analysis of Financial Condition and Results of Operations
    if section == 3 or section == 0:
        try:
            item7_start = re.compile("item\s*[7][\.\;\:\-\_]*\s*\\bM", re.IGNORECASE)
            item7_end = re.compile("item\s*7a[\.\;\:\-\_]\sQuanti|item\s*8[\.\,\;\:\-\_]\s*", re.IGNORECASE)
            mdaText = extract_text(text, item7_start, item7_end)
            return mdaText
        except:
            mdaText = "Something went wrong!"

    ### Section 9 Accounting
    if section == 9:
        try:
            item9_start = re.compile("item\s*[9][\.\;\:\-\_]*\s*Ch", re.IGNORECASE)
            item9_end = re.compile("item\s*9a[\.\;\:\-\_]\sControls|item\s*9b[\.\,\;\:\-\_]\s*", re.IGNORECASE)
            accText = extract_text(text, item9_start, item9_end)
            return accText
        except:
            accText = "Something went wrong!"

    ## Governance
    ## "Directors, Executive Officers and Corporate Governance": "10",
    if section == 10:
        try:
            item10_start = re.compile("item\s*^10$[\.\;\:\-\_]*\s*\\bDir", re.IGNORECASE)
            item10_end = re.compile("item\s*11[\.\;\:\-\_]\sComp|item\s*12[\.\,\;\:\-\_]\s*", re.IGNORECASE)
            govText = extract_text(text, item10_start, item10_end)
            return govText
        except:
            govText = "Something went wrong!"


    # "Legal Proceedings": "3",
    if section == 4:
        try:
            item4_start = re.compile("item\s*[3][\.\;\:\-\_]*\s*\\bLeg", re.IGNORECASE)
            item4_end = re.compile("item\s*4[\.\;\:\-\_]\sMin|item\s*5[\.\,\;\:\-\_]\s*", re.IGNORECASE)
            shareText = extract_text(text, item4_start, item4_end)
            return shareText
        except:
            shareText = "Something went wrong!"

    # Security Ownership of Certain Beneficial Owners and Management and Related Stockholder Matters
    if section == 12:
        try:
            item12_start = re.compile("item\s*\d{2}[\.\;\:\-\_]*\s*\\bSec", re.IGNORECASE)
            item12_end = re.compile("item\s*13[\.\;\:\-\_]\sRel|item\s*14[\.\,\;\:\-\_]\s*", re.IGNORECASE)
            secText = extract_text(text, item12_start, item12_end)
            return secText
        except:
            secText = "Something went wrong!"

In [344]:
file_paths = glob.glob(os.path.join(html_folder, "*.html")) + glob.glob(os.path.join(html_folder, "*.htm"))

In [346]:
def sec_data_func(full_text, section):
    data = section_text(full_text, section)
    data_n = re.sub(r'\n', ' ', str(data))
    cleaned_text = re.sub(r'[^A-Za-z0-9]', ' ', str(data_n))
    preprocessed_data = preprocess(str(cleaned_text))

    return preprocessed_data

In [388]:
# 1,2,3,9,10,12,4
section_texts = {}
for path in file_paths:
    try:
        name = path.split("10k")[1].split('/')[1].split('.html')[0]
        raw_html = read_html_file(path)
        full_text = html_to_text(raw_html)
        bus_ov = sec_data_func(full_text, 1)
        risk = sec_data_func(full_text, 2)
        fin = sec_data_func(full_text, 3)
        acc = sec_data_func(full_text, 9)
        gov = sec_data_func(full_text, 10)
        sec = sec_data_func(full_text, 12)
        legal = sec_data_func(full_text, 4)
        section_texts[name] = {"overview":bus_ov, "risk":risk, "financial":fin, "accounting":acc,
                            "governance":gov, "security":sec, "legal":legal}
    except:
        print("Exception Caused")

/var/folders/gk/9y2llpc11ds8x2x4crf34bfh0000gn/T/ipykernel_21174/2210426232.py:7: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


In [398]:
def simple_tokenize(text: str) -> List[str]:
    tokens = re.findall(r"\w+", text.lower())
    return tokens

def chunk_by_sentence_overlap(text: str, max_sentences: int = 6, overlap: int = 1) -> List[Tuple[int,int,str]]:
    sentences = re.split(r'(?<=[\.\?\!])\s+', text.strip())
    chunks = []
    i = 0
    chunk_id = 0
    while i < len(sentences):
        chunk_sentences = sentences[i:i+max_sentences]
        chunk_text = " ".join(chunk_sentences).strip()
        start = i
        end = i + len(chunk_sentences)
        chunks.append((chunk_id, start, end, chunk_text))
        chunk_id += 1
        i += max_sentences - overlap
    return chunks

def chunk_by_token_window(text: str, window_size: int = 300, overlap: int = 50) -> List[Tuple[int,int,int,str]]:
    tokens = re.findall(r"\w+|\S", text)  # crude tokenization that preserves punctuation tokens if any
    if not tokens:
        return []
    chunks = []
    start = 0
    chunk_id = 0
    while start < len(tokens):
        end = min(start + window_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = " ".join(chunk_tokens)
        chunks.append((chunk_id, start, end, chunk_text))
        chunk_id += 1
        if end == len(tokens):
            break
        start = end - overlap
    return chunks


def build_chunks_from_section_texts(section_texts: Dict[str, Dict[int, str]], chunk_method: str = "token",
                                    window_size: int = 300, overlap: int = 50,
                                    max_sentences: int = 6, sentence_overlap: int = 1):
    all_chunks = []
    for filename, sections in section_texts.items():
        for section_num, text in sections.items():
            if not text or text.strip() == "":
                continue
            if chunk_method == "sentence":
                chunks = chunk_by_sentence_overlap(text, max_sentences=max_sentences, overlap=sentence_overlap)
                for cid, sidx, eidx, chunk_text in chunks:
                    all_chunks.append({"doc_id": f"{filename}::sec{section_num}::chunk{cid}","filename": filename,
                                        "section": section_num, "chunk_id": cid, "start_sent": sidx, "end_sent": eidx,
                                        "text": chunk_text})
            else:
                chunks = chunk_by_token_window(text, window_size=window_size, overlap=overlap)
                for cid, start_tok, end_tok, chunk_text in chunks:
                    all_chunks.append({
                        "doc_id": f"{filename}::sec{section_num}::chunk{cid}",
                        "filename": filename,
                        "section": section_num,
                        "chunk_id": cid,
                        "start_token": start_tok,
                        "end_token": end_tok,
                        "text": chunk_text
                    })
    return all_chunks


class BM25Retriever:
    def __init__(self, chunk_docs: List[Dict], tokenizer=simple_tokenize):
        self.docs = chunk_docs
        self.tokenizer = tokenizer
        self.corpus_tokens = [self.tokenizer(d["text"]) for d in self.docs]
        self.bm25 = BM25Okapi(self.corpus_tokens)

    def search(self, query: str, top_k: int = 10):
        q_tokens = self.tokenizer(query)
        scores = self.bm25.get_scores(q_tokens)
        ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        results = []
        for rank, idx in enumerate(ranked_idx, start=1):
            d = self.docs[idx]
            results.append({
                "rank": rank,
                "doc_id": d["doc_id"],
                "filename": d["filename"],
                "section": d["section"],
                "chunk_id": d["chunk_id"],
                "score": float(scores[idx]),
                "text_snippet": d["text"][:600]  # return first 600 chars as preview
            })
        return results

if __name__ == "__main__":
    chunks = build_chunks_from_section_texts(section_texts,
                                             chunk_method="token",
                                             window_size=300,
                                             overlap=100)

    print(f"Built {len(chunks)} chunks total")

    import json
    with open("chunks_index.json", "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    retriever = BM25Retriever(chunks)

Built 111402 chunks total
